In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

tsr564_dpo_dataset_full_path = kagglehub.dataset_download('tsr564/dpo-dataset-full')

print('Data source import complete.')


In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

# ─── HuggingFace Auth ─────────────────────────────────────────────────────────
os.environ["HF_TOKEN"] = ""
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [ ]:
!pip install -U -q trl
!pip install -U -q bitsandbytes
!pip install -U -q accelerate
!pip install -U -q mistral-common
!pip install -U -q kagglehub
!pip install -U -q transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 8.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 7.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 36.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 112.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 80.0 MB/s eta 0:00:00:00:01:01


In [ ]:
import os
import json
import torch
from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    Mistral3ForConditionalGeneration,
    BitsAndBytesConfig,
)
from peft import LoraConfig, TaskType, PeftModel
from trl import DPOTrainer, DPOConfig

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# -------------------------------------------------------
# Configuration (T4 Safe)
# -------------------------------------------------------

MODEL_ID = "mistralai/Ministral-3-3B-Instruct-2512-BF16"
DATA_DIR = Path("/kaggle/input/datasets/tsr564/dpo-dataset-full")

OUTPUT_DIR = Path("./checkpoints/ministral3b_dpo")

MAX_PROMPT_LEN = 512
MAX_TOTAL_LEN = 768

BATCH_SIZE = 1
GRAD_ACCUM = 8

LR = 5e-5
NUM_EPOCHS = 2
BETA = 0.05

LORA_RANK = 16
LORA_ALPHA = 32

SAVE_STEPS = 50
EVAL_STEPS = 50

SEED = 42

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------
# dtype
# -------------------------------------------------------

BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
torch_dtype = torch.bfloat16 if BF16 else torch.float16

# -------------------------------------------------------
# 4bit config
# -------------------------------------------------------

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch_dtype,
)

# -------------------------------------------------------
# Dataset helpers
# -------------------------------------------------------

def load_records(path: Path):
    with open(path) as f:
        data = json.load(f)
    return data


def to_dpo_dataset(records, tokenizer):

    rows = []

    for rec in records:

        prompt_str = tokenizer.apply_chat_template(
            rec["prompt"],
            tokenize=False,
            add_generation_prompt=True,
        )

        rows.append(
            {
                "prompt": prompt_str,
                "chosen": rec["chosen"][0]["content"],
                "rejected": rec["rejected"][0]["content"],
            }
        )

    return Dataset.from_list(rows)


# -------------------------------------------------------
# main
# -------------------------------------------------------

def main():

    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        padding_side="left",
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # -------------------------
    # datasets
    # -------------------------

    print("Loading datasets...")

    train_dataset = to_dpo_dataset(
        load_records(DATA_DIR / "dpo_train.json"),
        tokenizer,
    )

    val_dataset = to_dpo_dataset(
        load_records(DATA_DIR / "dpo_val.json"),
        tokenizer,
    )

    print("Train:", len(train_dataset))
    print("Val:", len(val_dataset))

    # -------------------------
    # policy model
    # -------------------------

    print("Loading policy model...")

    model = Mistral3ForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        torch_dtype=torch_dtype,
        device_map="auto",
        low_cpu_mem_usage=True,
        attn_implementation="eager",
    )

    model.config.use_cache = False

    # -------------------------
    # LoRA
    # -------------------------

    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=0.05,
        bias="none",
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
    )

    # -------------------------
    # DPO config
    # -------------------------

    dpo_config = DPOConfig(
        output_dir=str(OUTPUT_DIR),

        num_train_epochs=NUM_EPOCHS,

        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,

        gradient_accumulation_steps=GRAD_ACCUM,

        learning_rate=LR,

        lr_scheduler_type="cosine",

        warmup_ratio=0.1,

        logging_steps=10,

        save_steps=SAVE_STEPS,

        eval_steps=EVAL_STEPS,

        beta=BETA,

        loss_type="sigmoid",

        # max_prompt_length=MAX_PROMPT_LEN,
        max_length=MAX_TOTAL_LEN,

        gradient_checkpointing=True,

        bf16=BF16,
        fp16=not BF16,

        seed=SEED,

        # VERY IMPORTANT
        # reduces GPU usage massively
        precompute_ref_log_probs=True,

        # DDP safety
        ddp_find_unused_parameters=False,
    )

    # -------------------------
    # trainer
    # -------------------------

    trainer = DPOTrainer(
        model=model,
        ref_model=None,   # ref handled via precompute
        args=dpo_config,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        peft_config=lora_config,
    )

    # -------------------------
    # train
    # -------------------------

    print("Starting training...")

    trainer.train()

    # -------------------------
    # save adapters
    # -------------------------

    print("Saving adapters...")

    adapter_path = OUTPUT_DIR / "lora_adapters"

    trainer.save_model(adapter_path)

    tokenizer.save_pretrained(adapter_path)

    # -------------------------
    # merge model
    # -------------------------

    print("Merging LoRA...")

    base = Mistral3ForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch_dtype,
        device_map="auto",
    )

    merged = PeftModel.from_pretrained(base, adapter_path)

    merged = merged.merge_and_unload()

    merged_path = OUTPUT_DIR / "merged"

    merged.save_pretrained(merged_path, safe_serialization=True)

    tokenizer.save_pretrained(merged_path)

    print("Training complete")


if __name__ == "__main__":
    main()

Loading tokenizer...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

Saving adapters...
Merging LoRA...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/458 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete


In [ ]:
!ls -r ./checkpoints/ministral3b_dpo

runs	   merged	  checkpoint-50   checkpoint-150
README.md  lora_adapters  checkpoint-158  checkpoint-100


In [ ]:
!pip install -q huggingface_hub

In [ ]:


from huggingface_hub import notebook_login
notebook_login()

In [ ]:

MODEL_PATH = "./checkpoints/ministral3b_dpo/merged"
REPO_ID = "pritam3355/ministral3b-dpo"

from transformers import Mistral3ForConditionalGeneration, AutoTokenizer

model = Mistral3ForConditionalGeneration.from_pretrained(MODEL_PATH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model.push_to_hub(REPO_ID)
tokenizer.push_to_hub(REPO_ID)

Loading weights:   0%|          | 0/458 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/31.0 [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/pritam3355/ministral3b-dpo/commit/1748204a0507953ebd2e76544ef3cf71b08f0b94', commit_message='Upload tokenizer', commit_description='', oid='1748204a0507953ebd2e76544ef3cf71b08f0b94', pr_url=None, repo_url=RepoUrl('https://huggingface.co/pritam3355/ministral3b-dpo', endpoint='https://huggingface.co', repo_type='model', repo_id='pritam3355/ministral3b-dpo'), pr_revision=None, pr_num=None)

In [ ]:
from transformers import Mistral3ForConditionalGeneration, AutoTokenizer

MODEL_PATH = "./checkpoints/ministral3b_dpo/merged"

model = Mistral3ForConditionalGeneration.from_pretrained(MODEL_PATH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

SHARDED_PATH = "./merged_sharded"

model.save_pretrained(
    SHARDED_PATH,
    safe_serialization=True,
    max_shard_size="2GB"   # key parameter
)

tokenizer.save_pretrained(SHARDED_PATH)


from huggingface_hub import HfApi

api = HfApi()

api.upload_folder(
    folder_path="./merged_sharded",
    repo_id="pritam3355/ministral3b-dpo",
    repo_type="model",
)

Loading weights:   0%|          | 0/458 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/4 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/pritam3355/ministral3b-dpo/commit/395bae9b80d971f1b024a3fb12bcc6b316fcf1a5', commit_message='Upload folder using huggingface_hub', commit_description='', oid='395bae9b80d971f1b024a3fb12bcc6b316fcf1a5', pr_url=None, repo_url=RepoUrl('https://huggingface.co/pritam3355/ministral3b-dpo', endpoint='https://huggingface.co', repo_type='model', repo_id='pritam3355/ministral3b-dpo'), pr_revision=None, pr_num=None)

Loading Model

In [ ]:
from transformers import Mistral3ForConditionalGeneration, AutoTokenizer

model = Mistral3ForConditionalGeneration.from_pretrained("pritam3355/ministral3b-dpo")
tokenizer = AutoTokenizer.from_pretrained("pritam3355/ministral3b-dpo")
model

model.safetensors:   0%|          | 0.00/7.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/458 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/126 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Mistral3ForConditionalGeneration(
  (model): Mistral3Model(
    (vision_tower): PixtralVisionModel(
      (patch_conv): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
      (ln_pre): PixtralRMSNorm((1024,), eps=1e-05)
      (transformer): PixtralTransformer(
        (layers): ModuleList(
          (0-23): 24 x PixtralAttentionLayer(
            (attention_norm): PixtralRMSNorm((1024,), eps=1e-05)
            (feed_forward): PixtralMLP(
              (gate_proj): Linear(in_features=1024, out_features=4096, bias=False)
              (up_proj): Linear(in_features=1024, out_features=4096, bias=False)
              (down_proj): Linear(in_features=4096, out_features=1024, bias=False)
              (act_fn): SiLUActivation()
            )
            (attention): PixtralAttention(
              (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
              (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
              (q_proj): Linear(in_f

In [ ]:
from transformers import Mistral3ForConditionalGeneration, AutoTokenizer
from peft import PeftModel

BASE_MODEL = "mistralai/Ministral-3-3B-Instruct-2512-BF16"
ADAPTER_PATH = "./checkpoints/ministral3b_dpo/lora_adapters"
REPO_ID = "pritam3355/ministral3b-dpo-lora"

base_model = Mistral3ForConditionalGeneration.from_pretrained(BASE_MODEL)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)

model.push_to_hub(REPO_ID)
tokenizer.push_to_hub(REPO_ID)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/458 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/31.0 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/pritam3355/ministral3b-dpo-lora/commit/8a89e58f03783d613495dfc2b4f6791fb948f262', commit_message='Upload tokenizer', commit_description='', oid='8a89e58f03783d613495dfc2b4f6791fb948f262', pr_url=None, repo_url=RepoUrl('https://huggingface.co/pritam3355/ministral3b-dpo-lora', endpoint='https://huggingface.co', repo_type='model', repo_id='pritam3355/ministral3b-dpo-lora'), pr_revision=None, pr_num=None)

In [ ]:
# import torch
# torch.cuda.empty_cache() # Release unused cached memory [2, 7]
# import gc
# gc.collect()      # Force garbage collection [5]


In [ ]:
import os
import json
import torch
from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    Mistral3ForConditionalGeneration,
    BitsAndBytesConfig,
)
from peft import LoraConfig, TaskType, PeftModel
from trl import DPOTrainer, DPOConfig

# ── GPU visibility ────────────────────────────────────────────────────────────
# Must be set BEFORE torch initialises CUDA.
# Hiding GPU 1 prevents Accelerate/DataParallel from splitting the model across
# devices, which causes illegal memory access in SDPA + gradient checkpointing.
os.environ["CUDA_VISIBLE_DEVICES"]      = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"]  = "expandable_segments:True"

# ─── Configuration ────────────────────────────────────────────────────────────
MODEL_ID        = "mistralai/Ministral-3-3B-Instruct-2512-BF16"
DATA_DIR        = Path("/kaggle/input/datasets/tsr564/dpo-dataset-full")
OUTPUT_DIR      = Path("./checkpoints/ministral3b_dpo")
MAX_PROMPT_LEN  = 1024
MAX_TOTAL_LEN   = 1280
BATCH_SIZE      = 1
GRAD_ACCUM      = 8     # effective batch = 16
LR              = 5e-5
NUM_EPOCHS      = 2
BETA            = 0.05
LORA_RANK       = 32
LORA_ALPHA      = 64
SAVE_STEPS      = 50
EVAL_STEPS      = 50
SEED            = 42
WARMUP_RATIO    = 0.10

# ─── Flash Attention (optional) ───────────────────────────────────────────────
# Set to True only if flash-attn is installed:
#   pip install flash-attn --no-build-isolation
# Supported on: Ampere+ GPUs (A100, A10, RTX 3090/4090).
# NOT supported on: T4, P100, V100 (Kaggle default GPUs).
USE_FLASH_ATTN  = False

LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

MISTRAL_EOS_TOKEN = "</s>"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ─── Device / dtype setup ─────────────────────────────────────────────────────
BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
torch_dtype = torch.bfloat16 if BF16 else torch.float16

if torch.cuda.is_available():
    # After CUDA_VISIBLE_DEVICES="0", only one device exists from PyTorch's view.
    # device_map="auto" now safely maps everything to that single visible GPU.
    device_map = "auto"
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    gpu_name = torch.cuda.get_device_name(0)
    print(f"Visible GPU      : {gpu_name} ({torch.cuda.get_device_properties(0).total_memory // 1024**3} GB)")
    print(f"bf16 support     : {BF16}  →  dtype={torch_dtype}")
    print(f"Flash Attention  : {'enabled' if USE_FLASH_ATTN else 'disabled'}")
else:
    device_map  = None
    torch_dtype = torch.float32
    print("No CUDA found — running on CPU (not recommended for DPO)")

# ─── Attention implementation ─────────────────────────────────────────────────
# "flash_attention_2" : best performance, requires flash-attn + Ampere+ GPU
# "eager"             : safest — no SDPA illegal memory access with BnB + grad ckpt
# "sdpa"              : faster than eager but has known illegal memory access bug
#                       when combined with BnB quantization + gradient checkpointing
if USE_FLASH_ATTN:
    ATTN_IMPL = "flash_attention_2"
else:
    ATTN_IMPL = "eager"   # ← NOT sdpa: avoids CUDA illegal memory access

# ─── BitsAndBytes 4-bit QLoRA config ─────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch_dtype,
    bnb_4bit_use_double_quant=True,
)

# ─── Dataset ──────────────────────────────────────────────────────────────────
def load_records(path: Path) -> list[dict]:
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, list):
        return data
    raise ValueError(f"Expected a JSON array in {path}, got {type(data)}")


def to_dpo_dataset(records: list[dict], tokenizer) -> Dataset:
    rows = []
    for rec in records:
        prompt_str = tokenizer.apply_chat_template(
            rec["prompt"],
            tokenize=False,
            add_generation_prompt=True,
        )
        rows.append({
            "prompt":   prompt_str,
            "chosen":   rec["chosen"][0]["content"],
            "rejected": rec["rejected"][0]["content"],
        })
    return Dataset.from_list(rows)

Visible GPU      : Tesla T4 (14 GB)
bf16 support     : True  →  dtype=torch.bfloat16
Flash Attention  : disabled


In [ ]:

# ─── Main ─────────────────────────────────────────────────────────────────────
def main():
    print(f"Loading tokenizer: {MODEL_ID}")
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        padding_side="left",
    )

    if tokenizer.eos_token is None:
        tokenizer.eos_token    = MISTRAL_EOS_TOKEN
        tokenizer.eos_token_id = tokenizer.convert_tokens_to_ids(MISTRAL_EOS_TOKEN)
        print(f"  eos_token was None → set to '{MISTRAL_EOS_TOKEN}' "
              f"(id={tokenizer.eos_token_id})")
    if tokenizer.pad_token is None:
        tokenizer.pad_token    = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
        print(f"  pad_token was None → set to eos_token ('{tokenizer.pad_token}')")

    print(f"  eos='{tokenizer.eos_token}' (id={tokenizer.eos_token_id}) | "
          f"pad='{tokenizer.pad_token}' (id={tokenizer.pad_token_id})")

    # ─── Datasets ────────────────────────────────────────────────────────────
    print("Loading DPO datasets...")
    train_recs    = load_records(DATA_DIR / "dpo_train.json")
    val_recs      = load_records(DATA_DIR / "dpo_val.json")
    train_dataset = to_dpo_dataset(train_recs, tokenizer)
    val_dataset   = to_dpo_dataset(val_recs,   tokenizer)
    print(f"  Train pairs: {len(train_dataset)} | Val pairs: {len(val_dataset)}")

    if len(train_dataset) < 20:
        print("  ⚠️  Very few DPO pairs.")

    steps_per_epoch = max(1, len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM))
    total_steps     = steps_per_epoch * NUM_EPOCHS
    warmup_steps    = max(1, int(total_steps * WARMUP_RATIO))
    print(f"  total_steps={total_steps} | warmup_steps={warmup_steps}")

    # ─── Policy model ────────────────────────────────────────────────────────
    print(f"\nLoading policy model (BnB NF4 4-bit) | attn={ATTN_IMPL} | dtype={torch_dtype}...")
    policy_model = Mistral3ForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        torch_dtype=torch_dtype,
        low_cpu_mem_usage=True,
        device_map=device_map,
        attn_implementation=ATTN_IMPL,
    )
    policy_model.config.use_cache = False

    # ─── Reference model (frozen, CPU only) ──────────────────────────────────
    print("Loading reference model (frozen, bf16, cpu)...")
    ref_model = Mistral3ForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch_dtype,
        low_cpu_mem_usage=True,
        device_map={"": "cpu"},
        attn_implementation=ATTN_IMPL,
    )
    for p in ref_model.parameters():
        p.requires_grad = False
    ref_model.eval()

    # ─── LoRA config ─────────────────────────────────────────────────────────
    lora_cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=0.05,
        target_modules=LORA_TARGET_MODULES,
        bias="none",
    )

    # ─── DPO training config ─────────────────────────────────────────────────
    dpo_config = DPOConfig(
        output_dir=str(OUTPUT_DIR),
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        optim="adamw_torch",
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_steps=warmup_steps,
        beta=BETA,
        loss_type="sigmoid",
        # max_prompt_length=MAX_PROMPT_LEN,
        max_length=MAX_TOTAL_LEN,
        bf16=BF16,
        fp16=not BF16,
        logging_steps=10,
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        eval_strategy="steps",
        eval_steps=EVAL_STEPS,
        load_best_model_at_end=True,
        metric_for_best_model="eval_rewards/accuracies",
        greater_is_better=True,
        save_total_limit=2,
        seed=SEED,
        logging_dir=str(OUTPUT_DIR / "tb_logs"),
        precompute_ref_log_probs=False,
    )

    # ─── Trainer ─────────────────────────────────────────────────────────────
    trainer = DPOTrainer(
        model=policy_model,
        ref_model=ref_model,
        args=dpo_config,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        peft_config=lora_cfg,
    )

    print("\nStarting DPO training...")
    print(f"  Model            : {MODEL_ID}")
    print(f"  Quantization     : BnB NF4 4-bit (policy) | bf16 cpu (ref)")
    print(f"  β={BETA} | loss=sigmoid | epochs={NUM_EPOCHS}")
    print(f"  Effective batch  : {BATCH_SIZE * GRAD_ACCUM}")
    print(f"  dtype            : {torch_dtype}  (bf16={BF16})")
    print(f"  Attention        : {ATTN_IMPL}")
    trainer.train()

    # ─── Log final metrics ───────────────────────────────────────────────────
    if trainer.state.log_history:
        eval_logs = [l for l in trainer.state.log_history
                     if "eval_rewards/accuracies" in l]
        if eval_logs:
            best = max(eval_logs, key=lambda x: x["eval_rewards/accuracies"])
            print(f"\n  Best eval reward accuracy: "
                  f"{best['eval_rewards/accuracies']:.4f} (step {best.get('step', '?')})")
        margin_logs = [l for l in trainer.state.log_history
                       if "eval_rewards/margins" in l]
        if margin_logs:
            print(f"  Final reward margin: {margin_logs[-1]['eval_rewards/margins']:.4f}")

    # ─── Save LoRA adapters ───────────────────────────────────────────────────
    print("\nSaving DPO LoRA adapters...")
    adapter_path = OUTPUT_DIR / "lora_adapters"
    trainer.save_model(str(adapter_path))
    tokenizer.save_pretrained(str(adapter_path))

    # ─── Merge LoRA → base weights ───────────────────────────────────────────
    print("\nMerging DPO LoRA into base weights...")
    base = Mistral3ForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch_dtype,
        device_map="auto",
        attn_implementation=ATTN_IMPL,
    )
    merged = PeftModel.from_pretrained(base, str(adapter_path))
    merged = merged.merge_and_unload()

    merged_path = OUTPUT_DIR / "merged"
    merged.save_pretrained(str(merged_path), safe_serialization=True)
    tokenizer.save_pretrained(str(merged_path))
    print(f"  Merged DPO model saved → {merged_path}")

    print("\nDPO complete. Next: python step4_5_distill.py --student both --cascade")


if __name__ == "__main__":
    main()

Loading tokenizer: mistralai/Ministral-3-3B-Instruct-2512-BF16


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

Tokenizing eval dataset:   0%|          | 0/173 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1, 'pad_token_id': 11}.



Starting DPO training...
  Model            : mistralai/Ministral-3-3B-Instruct-2512-BF16
  Quantization     : BnB NF4 4-bit (policy) | bf16 cpu (ref)
  β=0.05 | loss=sigmoid | epochs=2
  Effective batch  : 8
  dtype            : torch.bfloat16  (bf16=True)
  Attention        : eager


Step,Training Loss,Validation Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 462.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 415.81 MiB is free. Including non-PyTorch memory, this process has 14.15 GiB memory in use. Of the allocated memory 13.91 GiB is allocated by PyTorch, and 112.62 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)